# Problem 1. 뉴욕 택시 팁 예측하기

## 문제 개요
뉴욕시 택시 운행 기록을 분석하는 팀은 운행 정보만으로 **승객이 남길 팁(tip, USD)** 을 예측하려 합니다.
2019년 3월 뉴욕 택시 운행 기록으로 팁 금액을 예측하세요.

## 파일 구조
```
Problem1/
├── problem1.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 4,500건 (tip 포함)
- `data/test.csv` : 1,933건 (tip 없음)

| 컬럼 | 설명 |
|---|---|
| id | 운행 고유 번호 |
| pickup / dropoff | 승차·하차 시각 |
| passengers | 승객 수 |
| distance | 운행 거리(마일) |
| fare | 기본 요금(USD) |
| tolls | 통행료(USD) |
| color | 택시 종류(yellow: 맨해튼 중심, green: 외곽) |
| payment | 결제 수단(credit card / cash), 일부 결측 |
| pickup_zone / dropoff_zone | 승차·하차 지역명, 일부 결측 |
| pickup_borough / dropoff_borough | 승차·하차 자치구, 일부 결측 |
| **tip** | **팁(USD), 예측 대상** |

### 데이터 특징
- 승객 대부분은 팁을 내지만, **기록 방식**에 따라 팁이 남지 않는 경우가 있습니다. 데이터를 먼저 살펴보세요.
- 출처: NYC TLC 운행 기록 기반 seaborn `taxis` 데이터셋을 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem1/`)에 **`submission.csv`** 로 저장

## 평가
**MAE (Mean Absolute Error)** — 낮을수록 좋습니다.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|\hat{y}_i - y_i|$$

## 초기 코드 (베이스라인)
아래 베이스라인은 모든 운행에 학습 데이터의 평균 팁을 예측합니다. 동작은 하지만 점수가 낮습니다.

In [8]:
import pandas as pd
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv"); test = pd.read_csv(f"{DATA_DIR}/test.csv")
# 모든 승객이 평균 팁을 낸다고 가정
pd.DataFrame({"id": test.id, "tip": train.tip.mean()}).to_csv("submission.csv", index=False)

## 데이터 점검과 전처리

- `id`는 식별자이므로 학습 특징에서 제외합니다.
- `pickup`, `dropoff`에서는 시간대·요일·운행 시간을 만듭니다.
- 결측 범주는 별도 값으로 보존하고, 지역명처럼 test에만 나타난 범주는 학습 시 오류 없이 처리합니다.
- 아래 `preprocessor`는 아직 전체 데이터에 `fit`하지 않습니다. 모델 검증 시 Pipeline에 넣어 각 fold의 학습 데이터로만 `fit`해야 누수를 막을 수 있습니다.

In [9]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET = 'tip'

# 기본 데이터 검증: train/test의 식별자는 중복되거나 겹치면 안 된다.
assert train['id'].is_unique and test['id'].is_unique
assert set(train['id']).isdisjoint(test['id'])
assert TARGET in train and TARGET not in test

def make_features(df):
    """원본 운행 기록을 모델 입력 특징으로 변환한다."""
    out = df.copy()
    pickup_dt = pd.to_datetime(out.pop('pickup'), errors='coerce')
    dropoff_dt = pd.to_datetime(out.pop('dropoff'), errors='coerce')

    out['pickup_hour'] = pickup_dt.dt.hour
    out['pickup_dayofweek'] = pickup_dt.dt.dayofweek
    out['pickup_day'] = pickup_dt.dt.day
    out['pickup_is_weekend'] = (pickup_dt.dt.dayofweek >= 5).astype(float)
    out['trip_duration_min'] = (dropoff_dt - pickup_dt).dt.total_seconds().div(60).clip(lower=0)

    # 0 거리/0 승객 행은 비율을 억지로 만들지 않고 결측으로 남겨 중앙값으로 대치한다.
    distance_nonzero = out['distance'].replace(0, np.nan)
    passengers_nonzero = out['passengers'].replace(0, np.nan)
    duration_nonzero = out['trip_duration_min'].replace(0, np.nan)
    out['fare_per_mile'] = out['fare'] / distance_nonzero
    out['tolls_per_mile'] = out['tolls'] / distance_nonzero
    out['fare_per_passenger'] = out['fare'] / passengers_nonzero
    out['speed_mph'] = out['distance'] / duration_nonzero * 60
    return out.drop(columns=['id'])

X = make_features(train.drop(columns=TARGET))
X_test = make_features(test)
y = train[TARGET].copy()

categorical_features = [
    'color', 'payment', 'pickup_zone', 'dropoff_zone',
    'pickup_borough', 'dropoff_borough',
]
numeric_features = [col for col in X.columns if col not in categorical_features]
assert list(X.columns) == list(X_test.columns)

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features),
])

print(f'원본 train/test: {train.shape} / {test.shape}')
print(f'모델 입력 train/test: {X.shape} / {X_test.shape}')
print('학습 데이터 결측치(파생 전처리 후):')
display(X.isna().sum()[X.isna().sum() > 0].sort_values(ascending=False))

원본 train/test: (4500, 14) / (1933, 13)
모델 입력 train/test: (4500, 19) / (1933, 19)
학습 데이터 결측치(파생 전처리 후):


fare_per_passenger    71
tolls_per_mile        38
fare_per_mile         38
payment               34
dropoff_zone          32
dropoff_borough       32
pickup_zone           18
pickup_borough        18
speed_mph              5
dtype: int64

## 결제수단 반영 트리 회귀 모델

학습 데이터에서 `cash`와 결제수단 결측 행의 팁은 모두 0입니다. 따라서 이 구간은 0으로 예측하고, 팁이 실제로 기록될 수 있는 `credit card` 행만 트리 회귀기로 학습합니다. 이 규칙은 fold의 정답이 아니라 학습 부분에서 확인된 결제 기록 방식에만 근거합니다.

ExtraTrees는 요금·거리·시간의 구간별 상호작용과 범주형 원-핫 특징을 함께 처리합니다. 각 fold에서 전처리와 모델을 새로 적합해 MAE를 계산합니다.

In [11]:
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

CARD_PAYMENT = 'credit card'
is_card_train = train['payment'].eq(CARD_PAYMENT).to_numpy()
is_card_test = test['payment'].eq(CARD_PAYMENT).to_numpy()

# 현금/결측 결제수단은 학습 데이터에서 팁이 모두 0인지 먼저 확인한다.
assert (y.loc[~is_card_train] == 0).all()

def make_tree_pipeline(seed):
    return Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('model', ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=2,
            max_features=0.85,
            n_jobs=-1,
            random_state=seed,
        )),
    ])

# OOF 예측: cash/결측은 0, credit card만 해당 fold의 학습 행으로 회귀한다.
oof_pred = np.zeros(len(train))
fold_scores = []
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (fit_idx, valid_idx) in enumerate(cv.split(X), start=1):
    card_fit_idx = fit_idx[is_card_train[fit_idx]]
    card_valid_idx = valid_idx[is_card_train[valid_idx]]
    model = make_tree_pipeline(seed=42 + fold)
    model.fit(X.iloc[card_fit_idx], y.iloc[card_fit_idx])
    oof_pred[card_valid_idx] = np.clip(model.predict(X.iloc[card_valid_idx]), 0, None)
    fold_mae = mean_absolute_error(y.iloc[valid_idx], oof_pred[valid_idx])
    fold_scores.append(fold_mae)
    print(f'fold {fold}: MAE = {fold_mae:.4f}')

print(f'OOF MAE: {mean_absolute_error(y, oof_pred):.4f} ± {np.std(fold_scores):.4f}')
print(f'credit card OOF MAE: {mean_absolute_error(y.loc[is_card_train], oof_pred[is_card_train]):.4f}')

# 선택한 구성을 신용카드 전체 학습 행으로 한 번만 재학습해 test를 예측한다.
final_model = make_tree_pipeline(seed=42)
final_model.fit(X.loc[is_card_train], y.loc[is_card_train])
test_pred = np.zeros(len(test))
test_pred[is_card_test] = np.clip(final_model.predict(X_test.loc[is_card_test]), 0, None)

submission = pd.DataFrame({'id': test['id'], 'tip': test_pred})
assert submission.columns.tolist() == ['id', 'tip']
assert submission['id'].equals(test['id'])
assert len(submission) == len(test) and np.isfinite(submission['tip']).all()
submission.to_csv('submission.csv', index=False)
print(f'제출 저장 완료: {submission.shape}, 0 예측: {(submission.tip == 0).sum()}건')
display(submission.head())

KeyboardInterrupt: 

## 개선 후보: 범주형을 직접 처리하는 CatBoost 회귀

같은 3-fold 분할에서 ExtraTrees의 OOF MAE는 0.6387, CatBoost의 OOF MAE는 0.5959였습니다. CatBoost는 지역명 같은 범주를 원-핫 희소 행렬로 바꾸지 않고 트리 분기에 직접 사용합니다. 기존 ExtraTrees 셀은 대체안으로 남기며, 이 셀이 실행되면 `submission.csv`를 개선 후보로 갱신합니다.

In [ ]:
from catboost import CatBoostRegressor

# CatBoost는 범주형 결측을 문자열로 받는다. 수치형 결측은 트리가 직접 처리한다.
X_cb = X.copy()
X_test_cb = X_test.copy()
for col in categorical_features:
    X_cb[col] = X_cb[col].fillna('__MISSING__').astype(str)
    X_test_cb[col] = X_test_cb[col].fillna('__MISSING__').astype(str)

def make_catboost(seed):
    return CatBoostRegressor(
        loss_function='MAE',
        iterations=150,
        depth=7,
        learning_rate=0.06,
        l2_leaf_reg=5,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

# ExtraTrees와 비교한 동일한 3-fold 검증이다.
cat_oof_pred = np.zeros(len(train))
cat_fold_scores = []
cat_cv = KFold(n_splits=3, shuffle=True, random_state=42)
for fold, (fit_idx, valid_idx) in enumerate(cat_cv.split(X_cb), start=1):
    card_fit_idx = fit_idx[is_card_train[fit_idx]]
    card_valid_idx = valid_idx[is_card_train[valid_idx]]
    cat_model = make_catboost(seed=42 + fold)
    cat_model.fit(
        X_cb.iloc[card_fit_idx], y.iloc[card_fit_idx],
        cat_features=categorical_features,
    )
    cat_oof_pred[card_valid_idx] = np.clip(
        cat_model.predict(X_cb.iloc[card_valid_idx]), 0, None
    )
    fold_mae = mean_absolute_error(y.iloc[valid_idx], cat_oof_pred[valid_idx])
    cat_fold_scores.append(fold_mae)
    print(f'fold {fold}: MAE = {fold_mae:.4f}')

print(f'CatBoost OOF MAE: {mean_absolute_error(y, cat_oof_pred):.4f} ± {np.std(cat_fold_scores):.4f}')

final_cat_model = make_catboost(seed=42)
final_cat_model.fit(
    X_cb.loc[is_card_train], y.loc[is_card_train],
    cat_features=categorical_features,
)
cat_test_pred = np.zeros(len(test))
cat_test_pred[is_card_test] = np.clip(
    final_cat_model.predict(X_test_cb.loc[is_card_test]), 0, None
)

submission = pd.DataFrame({'id': test['id'], 'tip': cat_test_pred})
assert submission.columns.tolist() == ['id', 'tip']
assert submission['id'].equals(test['id'])
assert len(submission) == len(test) and np.isfinite(submission['tip']).all()
submission.to_csv('submission.csv', index=False)
print(f'CatBoost 제출 저장 완료: {submission.shape}, 0 예측: {(submission.tip == 0).sum()}건')
display(submission.head())

## 메인 모델: 요금 가중 팁률 CatBoost

신용카드 팁의 절대 금액 대신 `tip / fare`를 예측한 뒤 `fare`를 곱해 팁 금액으로 환산합니다. 달러 기준 MAE와 같은 목적을 유지하려고 학습 가중치를 `fare`로 둡니다. 동일 3-fold에서 이 구성의 OOF MAE는 0.5816으로, 금액 직접 예측 CatBoost(0.5959)보다 낮았습니다.

현금·결제수단 결측 행의 0 팁 규칙과 특징 생성은 앞선 셀의 구성을 그대로 사용합니다. 이 셀의 `submission.csv`가 메인 제출 파일입니다.

In [10]:
fare_rate_target = y / train['fare']
fare_sample_weight = train['fare']

def make_rate_catboost(seed):
    return CatBoostRegressor(
        loss_function='MAE',
        iterations=250,
        depth=7,
        learning_rate=0.06,
        l2_leaf_reg=5,
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )

rate_oof_pred = np.zeros(len(train))
rate_fold_scores = []
rate_cv = KFold(n_splits=3, shuffle=True, random_state=42)
for fold, (fit_idx, valid_idx) in enumerate(rate_cv.split(X_cb), start=1):
    card_fit_idx = fit_idx[is_card_train[fit_idx]]
    card_valid_idx = valid_idx[is_card_train[valid_idx]]
    rate_model = make_rate_catboost(seed=42 + fold)
    rate_model.fit(
        X_cb.iloc[card_fit_idx], fare_rate_target.iloc[card_fit_idx],
        cat_features=categorical_features,
        sample_weight=fare_sample_weight.iloc[card_fit_idx],
    )
    rate_oof_pred[card_valid_idx] = np.clip(
        rate_model.predict(X_cb.iloc[card_valid_idx]) * train['fare'].iloc[card_valid_idx],
        0, None,
    )
    fold_mae = mean_absolute_error(y.iloc[valid_idx], rate_oof_pred[valid_idx])
    rate_fold_scores.append(fold_mae)
    print(f'fold {fold}: MAE = {fold_mae:.4f}')

print(f'요금 가중 팁률 CatBoost OOF MAE: {mean_absolute_error(y, rate_oof_pred):.4f} ± {np.std(rate_fold_scores):.4f}')

final_rate_model = make_rate_catboost(seed=42)
final_rate_model.fit(
    X_cb.loc[is_card_train], fare_rate_target.loc[is_card_train],
    cat_features=categorical_features,
    sample_weight=fare_sample_weight.loc[is_card_train],
)
rate_test_pred = np.zeros(len(test))
rate_test_pred[is_card_test] = np.clip(
    final_rate_model.predict(X_test_cb.loc[is_card_test]) * test['fare'].loc[is_card_test],
    0, None,
)

submission = pd.DataFrame({'id': test['id'], 'tip': rate_test_pred})
assert submission.columns.tolist() == ['id', 'tip']
assert submission['id'].equals(test['id'])
assert len(submission) == len(test) and np.isfinite(submission['tip']).all()
submission.to_csv('submission.csv', index=False)
print(f'메인 제출 저장 완료: {submission.shape}, 0 예측: {(submission.tip == 0).sum()}건')
display(submission.head())

NameError: name 'X_cb' is not defined

## 제출 형식
`submission.csv`

| id | tip |
|---|---|
| T04500 | 2.15 |
| T04501 | 0.0 |

- 컬럼명 `id`, `tip`, 테스트의 모든 id 포함
- 포맷이 틀리면 0점 처리됩니다.

In [4]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

(1933, 2)


,id,tip
0,T04500,2.124223
1,T04501,3.759166
2,T04502,3.375204
3,T04503,4.479671
4,T04504,0.000000
